# Геометрическая калибровка

Две прямые оцениваются независимо для каждого сетапа. Остатки — перпендикулярные расстояния в пикселях. Пара проходит коридор, когда обе точки находятся на расстоянии не более выбранной полуширины.

**In-sample** оценивает прямые на данных подгонки. **Out-of-sample** оценивает прямые на остальных размеченных парах того же сетапа. Интервалы на графиках показывают 5-й и 95-й перцентили успешных повторений.

In [ ]:
import ipywidgets as widgets
import plotly.io as pio
from IPython.display import display

from laser_detection.calibration_experiments import run_calibration_study
from laser_detection.calibration_plots import (
    calibration_figures, inspect_calibration_image, largest_residuals, spatial_calibration,
)
from laser_detection.config import load_config
from laser_detection.data import load_dataset
from laser_detection.export import export_calibration
from laser_detection.geometry import add_geometry

pio.renderers.default = "plotly_mimetype"

config = load_config()
frame = add_geometry(load_dataset(config))
study = run_calibration_study(frame, config)
figures = calibration_figures(study, config)

## Подгонка на всех размеченных парах

In [ ]:
display(study.summary)
figures["residual_ecdf"].show()
figures["corridor_coverage"].show()

## Пространственная карта и кадры с наибольшими остатками

Крестики отмечают выбросы относительно порога RANSAC. Пунктирные границы и столбец `passes_corridor` используют выбранную полуширину коридора. Выберите кадр из таблицы для просмотра фотографии и прямых.

In [ ]:
setups = study.parameters["setup_id"].tolist()
thresholds = config.calibration.corridor_thresholds
setup_picker = widgets.Dropdown(options=setups, description="Сетап")
width_picker = widgets.Dropdown(
    options=thresholds, value=20 if 20 in thresholds else thresholds[0], description="± пикселей",
)
frame_picker = widgets.Dropdown(description="Кадр")
show_frame = widgets.Button(description="Показать кадр")
map_output = widgets.Output()
frame_output = widgets.Output()


def show_setup(_=None):
    setup, threshold = setup_picker.value, width_picker.value
    rows = largest_residuals(study, setup, threshold)
    frame_picker.options = rows["image"].tolist()
    show_frame.disabled = rows.empty
    frame_output.outputs = ()
    map_output.outputs = ()
    map_output.append_display_data(spatial_calibration(study, setup, config, threshold))
    map_output.append_display_data(rows)


def show_image(_):
    frame_output.outputs = ()
    figure, details, message = inspect_calibration_image(frame, frame_picker.value, study, config)
    if message:
        frame_output.append_stdout(message + "\n")
    if details is not None:
        frame_output.append_display_data(details.to_frame("Значение"))
    if figure is not None:
        frame_output.append_display_data(figure)


setup_picker.observe(show_setup, names="value")
width_picker.observe(show_setup, names="value")
show_frame.on_click(show_image)
display(widgets.HBox([setup_picker, width_picker]), map_output, frame_picker, show_frame, frame_output)
show_setup()
if not config.images_dir.is_dir():
    print("Каталог фотографий недоступен. Таблицы и графики остатков доступны.")

## Ограниченный объём калибровки

Удержанные пары полностью участвуют в оценке. Пропуски и неудачные подгонки отображаются отдельно, а также учитываются в агрегатах.

In [ ]:
figures["sample_size_residual"].show()
figures["sample_size_coverage"].show()
sample_issues = study.sample_runs.loc[study.sample_runs["status"].ne("ok")]
display(sample_issues.groupby(
    ["setup_id", "training_size", "status", "message"], sort=False,
).size().rename("entries").reset_index())

## Синтетические ложные пары

Для каждого повторения используется один и тот же состав обучающих и удержанных кадров для всех долей загрязнения. Случайные координаты заменяют целые пары только в обучающей копии.

In [ ]:
figures["contamination_residual"].show()
display(study.contamination_summary.loc[study.contamination_summary["metric"].eq("test_pair_median")])
noise_issues = study.contamination_runs.loc[study.contamination_runs["status"].ne("ok")]
display(noise_issues.groupby(
    ["setup_id", "training_size", "contamination_fraction", "status", "message"], sort=False,
).size().rename("entries").reset_index())

## Локальный экспорт

Установите `EXPORT = True`, чтобы сохранить таблицы, графики и параметры эксперимента. Папка по умолчанию — `results/calibration/` около файла конфигурации. Перед коммитом очистите выводы ноутбука.

In [ ]:
EXPORT = False
if EXPORT:
    for path in export_calibration(study, figures, config):
        print(path)